# 🧠 CerebroSpheroid V4: 1,000-Step Scratch Training & Core Weight Diagnostic
### Continuous Concentric Manifold with Multi-Head RoPE & Causal Autoregressive KV Memory

This notebook executes a targeted **1,000-step training run from scratch** to:
1. **Train From Scratch**: Initialize weights cleanly at theoretical baseline ($\mathcal{L} \approx \ln(V)$).
2. **Track Weight Dynamics**: Observe how gradients shape the sensory vocabulary, multi-head arcuate fasciculus attention, and the volumetric core.
3. **Core Injection & Verification**: Evaluate how the Volumetric Core circulates linguistic flux, verifying that engaging subconscious deliberation maintains or lowers the loss.
4. **Autonomous Generation**: Validate that the new Causal KV cache eliminates the earlier repetition loops.


## Step 1: GPU Hardware Check
Confirm NVIDIA GPU availability and CUDA acceleration.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


## Step 2: Clone Latest Repository & Install Dependencies
Pulls the latest commit with the fixed Multi-Head Attention, RoPE tables, and Causal KV cache.

In [ ]:
import os
import sys

# Clone or pull latest repository
if not os.path.exists("/content/reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git /content/reasoning_agent
    %cd /content/reasoning_agent
else:
    %cd /content/reasoning_agent
    !git pull origin main

!pip install -q transformers datasets accelerate sentencepiece matplotlib pyarrow


## Step 3: Run Architecture & Scratch Convergence Verification
Verifies the parameter count (118,534,071), RoPE tables, and scratch loss descent.

In [ ]:
!python3 training/test_v4_architecture.py
!python3 training/test_v4_scratch_training.py


## Step 4: 1,000-Step Precision Trainer with Live Weight Tracking
Trains the fresh model for 1,000 steps with:
- Elastic context window
- Scheduled internal deliberation
- Live gradient norm and weight norm logging
- Matplotlib loss curve visualization


In [ ]:
import os
import sys
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT_DIR = "/content/reasoning_agent"
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.spheroid_v4 import build_spheroid_v4, compute_manifold_diagnostics
from src.corpus import build_tokenizer
from training.train_v4_full import build_batch
from datasets import load_dataset

dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {dev}")

# 1. Tokenizer
tokenizer = build_tokenizer("llama")
pad_id = tokenizer.pad_token_id or 0
vocab_size = len(tokenizer)

# 2. Build Fresh V4 Model (v4_small, 118.5M params)
model = build_spheroid_v4("v4_small", vocab_size=vocab_size).to(dev)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Initialized fresh CerebroSpheroid V4: {total_params:,} parameters")

# 3. Stream Training Data (OpenHermes-2.5)
print("📥 Loading OpenHermes-2.5 streaming dataset...")
ds = load_dataset("teknium/OpenHermes-2.5", split="train", streaming=True)
data_iter = iter(ds)

# 4. Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-5)

# 5. Training Loop Tracking
steps, losses, grad_norms, embed_norms, q_norms = [], [], [], [], []

print("=" * 80)
print("🚀 STARTING 1,000-STEP SCRATCH TRAINING RUN")
print("=" * 80)

model.train()
t0 = time.time()
batch_size = 4
target_len = 32

for step in range(1, 1001):
    # Collect batch
    pairs = []
    while len(pairs) < batch_size:
        item = next(data_iter)
        convs = item.get("conversations", [])
        if len(convs) >= 2:
            prompt = convs[0].get("value", "")
            answer = convs[1].get("value", "")
            if len(prompt.strip()) > 5 and len(answer.strip()) > 5:
                pairs.append(("openhermes", prompt, answer))
    
    batch = build_batch(tokenizer, pairs, window=target_len, rng=__import__("random").Random(step),
                        max_prompt_len=128, max_answer_tokens=target_len, pad_token_id=pad_id)
    if batch is None:
        continue
    p_t, t_t, f_t = batch
    p_t, t_t, f_t = p_t.to(dev), t_t.to(dev), f_t.to(dev)

    optimizer.zero_grad()
    
    # Scheduled deliberation: gradually ramp up internal thinking cycles
    delib_cycles = 1 if step < 300 else 3
    sampling_p = max(0.35, 1.0 - (step / 1000.0) * 0.65)
    
    logits = model(
        p_t,
        response_steps=t_t.size(1),
        thinking_cycles=delib_cycles,
        targets=t_t,
        feedback_tokens=f_t,
        deliberation_mode="scheduled",
        sampling_p=sampling_p,
        pad_token_id=pad_id,
    )
    
    loss = F.cross_entropy(logits.view(-1, vocab_size), t_t.view(-1), ignore_index=-100)
    loss.backward()
    
    # Compute total gradient norm
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    # Track metrics
    if step % 20 == 0 or step == 1:
        e_norm = model.sensory_dnn.embedding.weight.norm().item()
        q_norm = model.motor_dnn.q_proj.weight.norm().item()
        
        steps.append(step)
        losses.append(loss.item())
        grad_norms.append(total_norm.item())
        embed_norms.append(e_norm)
        q_norms.append(q_norm)
        
        elapsed = time.time() - t0
        tok_sec = (step * batch_size * target_len) / max(1, elapsed)
        print(f"Step {step:04d}/1000 | Loss: {loss.item():.4f} | GradNorm: {total_norm.item():.3f} | EmbedNorm: {e_norm:.1f} | LR: {scheduler.get_last_lr()[0]:.2e} | {tok_sec:.1f} tok/s")

# Save Checkpoint at Step 1000
os.makedirs("checkpoints", exist_ok=True)
ckpt_path = "checkpoints/cerebro_spheroid_v4_step_1000.pt"
model.save_compatible_checkpoint(ckpt_path, step=1000, best=min(losses), loss=losses[-1])
print(f"\n💾 Checkpoint saved to: {ckpt_path}")

# Plot Training Dynamics
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(steps, losses, "b-", lw=2, label="Cross-Entropy Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("V4 Scratch Training Loss (1,000 Steps)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(steps, grad_norms, "r-", lw=1.5, label="Gradient Norm")
plt.xlabel("Step")
plt.ylabel("Norm")
plt.title("Gradient Stability")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Step 5: Core Engagement Diagnostic
Inspects weight statistics and evaluates the loss impact of engaging the Volumetric Core:
- Tests loss with 0 core circulation cycles (shallow baseline)
- Tests loss with 1, 2, and 3 core circulation cycles
- Confirms that the core maintains or improves cross-entropy loss without disruption.

In [ ]:
model.eval()

print("=" * 80)
print("🔬 STEP 1,000 WEIGHT & CORE COUPLING DIAGNOSTIC")
print("=" * 80)

# 1. Weight Distributions
print("\n📊 Active Weight Norms & Dimensions:")
for name, p in model.named_parameters():
    if any(k in name for k in ["embedding", "intra_shell", "reentrant", "shortcut", "q_proj", "k_proj", "v_proj", "head"]):
        print(f"  {name:<38} | Shape: {str(list(p.shape)):<18} | Norm: {p.norm().item():.3f} | Mean: {p.mean().item():+.5f} | Std: {p.std().item():.5f}")

# 2. Test Core Engagement Impact on Held-Out Records
test_pairs = [
    ("openhermes", "What is the capital of France?", "The capital of France is Paris."),
    ("openhermes", "What is 2 + 3?", "2 + 3 is 5."),
    ("openhermes", "Why is the ocean salty?", "The ocean is salty due to dissolved minerals from rocks and volcanic vents."),
    ("openhermes", "Name three primary colors.", "The three primary colors are red, blue, and yellow."),
]

test_batch = build_batch(tokenizer, test_pairs, window=32, rng=__import__("random").Random(1337),
                         max_prompt_len=128, max_answer_tokens=32, pad_token_id=pad_id)
p_t, t_t, f_t = [t.to(dev) for t in test_batch]

print("\n🧠 Evaluating Loss Across Subconscious Core Thinking Cycles:")
for cycles in [0, 1, 2, 3, 4]:
    with torch.no_grad():
        out_logits = model(p_t, response_steps=t_t.size(1), thinking_cycles=cycles, targets=t_t, deliberation_mode="teacher_forcing")
        ce = F.cross_entropy(out_logits.view(-1, vocab_size), t_t.view(-1), ignore_index=-100).item()
        print(f"  Thinking Cycles = {cycles} | Cross-Entropy Loss: {ce:.4f} (Perplexity: {math.exp(ce):.2f})")

# 3. Manifold Shell Diagnostics
state = model.ingest_stimulus(p_t, pad_token_id=pad_id)
diag = compute_manifold_diagnostics(model, state)
print(f"\n🌐 Volumetric Manifold Effective Rank: {diag['mean_effective_rank']:.3f} | Shell Energy RMS: {diag['mean_shell_rms']:.4f}")


## Step 6: Test Autoregressive Free Generation
Validates that the causal KV cache allows the model to verbalize smoothly without repetition loops.

In [ ]:
test_prompts = [
    "What is the capital of France?",
    "Explain why the sky is blue in simple words.",
    "What is 2 + 3?",
    "Write a short sentence about artificial intelligence.",
    "If you have 5 apples and give 2 to a friend, how many do you have left?",
]

print("=" * 80)
print("📝 STEP 1,000 AUTOREGRESSIVE GENERATION (Active Causal KV Cache)")
print("=" * 80)

for prompt in test_prompts:
    response = model.generate_text(
        prompt,
        tokenizer=tokenizer,
        thinking_cycles=3,
        max_new_tokens=48,
        temperature=0.6,
        repetition_penalty=1.2,
        device=dev,
        strip_bio_prefix=True,
    )
    print(f"\nUser:      {prompt}")
    print(f"Spheroid:  {response}")
    print("-" * 60)


## Step 7: Export Checkpoint to Google Drive (Optional)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -v checkpoints/cerebro_spheroid_v4_step_1000.pt /content/drive/MyDrive/
    print("✅ Checkpoint backed up to Google Drive!")
except Exception as e:
    print("Google Drive mount skipped or not running on Colab.")
